# Marketing Response Optimization (Response Model)

This notebook implements a machine learning model to predict customer responses to marketing campaigns using the Bank Marketing Dataset from Kaggle.

## 1. Import Libraries and Load Dataset

Import pandas, scikit-learn, and visualization libraries; load the Bank Marketing dataset from a local Kaggle download or CSV file.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import opendatasets as od
import joblib

# Download the dataset from Kaggle
# Note: You need to have a Kaggle API key set up for this to work automatically.
# Alternatively, download the dataset manually from https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset
# and place the CSV file in the data/ folder.

od.download("https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset", data_dir='data/')

# Load the dataset
df = pd.read_csv('data/bank-marketing-dataset/bank.csv')
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:Your Kaggle username:

## 2. Explore Dataset and Preprocess

Inspect data shape, missing values, and class distribution; encode categorical variables, scale features, and split into training and test sets.

In [ ]:
# Explore the dataset
print("Dataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nClass Distribution:")
print(df['deposit'].value_counts())

# Visualize class distribution
sns.countplot(x='deposit', data=df)
plt.title('Class Distribution')
plt.show()

# Encode categorical variables
le = LabelEncoder()
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'deposit']
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# Split features and target
X = df.drop('deposit', axis=1)
y = df['deposit']

# Scale numerical features
scaler = StandardScaler()
numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## 3. Build Marketing Response Model

Train classification models such as logistic regression, random forest, or XGBoost to predict customer response, and compare performance.

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]

# Train Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Performance:")
print(f"Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print(f"Precision: {precision_score(y_test, lr_pred):.4f}")
print(f"Recall: {recall_score(y_test, lr_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, lr_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, lr_proba):.4f}")

print("\nRandom Forest Performance:")
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"Recall: {recall_score(y_test, rf_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, rf_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, rf_proba):.4f}")

# Choose the best model (Random Forest in this case)
best_model = rf_model

## 4. Evaluate Model Performance

Compute accuracy, precision, recall, F1 score, and ROC AUC; visualize results with confusion matrix and performance curves.

In [ ]:
# Confusion Matrix for Random Forest
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Classification Report
print("Classification Report:")
print(classification_report(y_test, rf_pred))

# Feature Importance
feature_importance = pd.DataFrame({'feature': X.columns, 'importance': rf_model.feature_importances_})
feature_importance = feature_importance.sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance.head(10))
plt.title('Top 10 Feature Importances')
plt.show()

## 5. Save Model and Deployment Considerations

Serialize the trained model with joblib or pickle; discuss how to load and use the model for batch predictions.

In [ ]:
# Save the model
joblib.dump(best_model, 'src/marketing_response_model.pkl')
joblib.dump(scaler, 'src/scaler.pkl')
joblib.dump(le, 'src/label_encoder.pkl')  # Note: This is simplistic; in practice, handle multiple encoders

print("Model and preprocessors saved successfully!")

# Example of loading and using the model
# loaded_model = joblib.load('src/marketing_response_model.pkl')
# loaded_scaler = joblib.load('src/scaler.pkl')
# # Preprocess new data similarly
# # prediction = loaded_model.predict(new_data)

## 6. Front-End Integration Decision

Explain whether a front end is useful for this project, and suggest options like Streamlit or Flask for a simple interface versus keeping it as an analytical notebook.

### Should You Use a Front-End?

Yes, a front-end is highly recommended for this project to make the model accessible and demonstrate its practical application. The Marketing Response Model can be used by marketing teams to predict customer responses in real-time, aiding in campaign optimization.

**Options:**
- **Streamlit:** Simple and fast for prototyping. Create an interactive web app where users can input customer data and get predictions.
- **Flask/Django:** More robust for production, allowing API endpoints for predictions.
- **Keep as Notebook:** Suitable for analysis and experimentation, but not for end-users.

For this project, we'll create a simple Streamlit app as an example.